In [2]:
#r "nuget: ScottPlot, 5.0.21"
#r "bin/Debug/net10.0/task14.dll"
#r "nuget: SkiaSharp.NativeAssets.Linux.NoDependencies, 2.88.7"
using System;
using System.Diagnostics;
using System.Collections.Generic;
using task14;

Installed Packages ScottPlot, 5.0.21 SkiaSharp.NativeAssets.Linux.NoDependencies, 2.88.7

Loading extensions from `/home/koteyka/.nuget/packages/skiasharp/2.88.7/interactive-extensions/dotnet/SkiaSharp.DotNet.Interactive.dll`

In [3]:
double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
Func<double, double> sinFunc = x => Math.Sin(x);

Console.WriteLine("Размер шага | Результат интеграла");
Console.WriteLine("---------------------------------");
foreach (var step in steps)
{
    double res = DefiniteIntegral.SolveSingleThread(-100, 100, sinFunc, step);
    Console.WriteLine($"{step:E1}        | {res:F6}");
}

Размер шага | Результат интеграла
---------------------------------
1,0E-001        | -0,000000
1,0E-002        | -0,000000
1,0E-003        | -0,000000
1,0E-004        | 0,000000
1,0E-005        | -0,000000
1,0E-006        | 0,000000


In [5]:
double optimalStep = 1e-5;
int[] threadCounts = { 1, 2, 4, 8, 12, 16 };
List<double> times = new List<double>();

foreach (int threads in threadCounts)
{
    List<double> localTimes = new List<double>();
    for (int i = 0; i < 5; i++)
    {
        Stopwatch sw = Stopwatch.StartNew();
        DefiniteIntegral.Solve(-100, 100, sinFunc, optimalStep, threads);
        sw.Stop();
        localTimes.Add(sw.Elapsed.TotalMilliseconds);
    }
    double averageTime = localTimes.Average();
    times.Add(averageTime);
    Console.WriteLine($"Потоков: {threads} | Время: {averageTime:F2} мс");
}

Stopwatch swSingle = Stopwatch.StartNew();
DefiniteIntegral.SolveSingleThread(-100, 100, sinFunc, optimalStep);
swSingle.Stop();
double singleThreadTime = swSingle.Elapsed.TotalMilliseconds;
Console.WriteLine($"\nЧестный однопоток (без Task/Thread): {singleThreadTime:F2} мс");

var plot = new ScottPlot.Plot();

double[] xs = threadCounts.Select(t => (double)t).ToArray();
double[] ys = times.ToArray();

var scatter = plot.Add.Scatter(xs, ys);
scatter.LineWidth = 2;
scatter.MarkerSize = 8;

string linuxFont = "Liberation Sans";

plot.Title("Время вычисления от количества потоков", size: 16);
plot.XLabel("Количество потоков", size: 12);
plot.YLabel("Время (мс)", size: 12);

ScottPlot.Fonts.Default = linuxFont;

plot.SavePng("performance_graph.png", 800, 500);

Потоков: 1 | Время: 663,88 мс
Потоков: 2 | Время: 328,49 мс
Потоков: 4 | Время: 189,26 мс
Потоков: 8 | Время: 125,88 мс
Потоков: 12 | Время: 101,66 мс
Потоков: 16 | Время: 111,02 мс

Честный однопоток (без Task/Thread): 611,10 мс
